In [14]:
import warnings
import numpy as np
# import lightgbm as lgb
# from fontTools.misc.cython import returns
# from pyarrow.types import is_large_binary
# from sympy.codegen.ast import continue_
# from xgboost import XGBRegressor
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.svm import SVR
# from sklearn.neural_network import MLPRegressor
# from sklearn.tree import DecisionTreeRegressor
# from statsmodels.tools.eval_measures import rmse, hqic_sigma
import pandas as pd
import re
# from optimize_params import *
import matplotlib.pyplot as plt
from datetime import datetime

# np.random.seed(42)

warnings.filterwarnings("ignore", category=RuntimeWarning)

def mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE)

    Parameters:
        y_true (array-like): Actual values
        y_pred (array-like): Predicted values

    Returns:
        float: MAPE in percentage (%)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [15]:
train = pd.read_parquet("data/gold/train.parquet")
val = pd.read_parquet("data/gold/val.parquet")
test = pd.read_parquet("data/gold/test.parquet")

train = train.reset_index(drop=True)
val = val.reset_index(drop=True)
test = test.reset_index(drop=True)

In [16]:
y_col = 'Sum of кВт'

In [17]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.6.0.dev20241112+cu121
12.1
True


In [18]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer
from sklearn.metrics import mean_squared_error

y_col = "Sum of кВт"

train = pd.read_parquet("data/gold/train.parquet").reset_index(drop=True)
val = pd.read_parquet("data/gold/val.parquet").reset_index(drop=True)
test = pd.read_parquet("data/gold/test.parquet").reset_index(drop=True)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


smape_scorer = make_scorer(
    name="SMAPE",
    score_func=smape,
    optimum=0,
    greater_is_better=False
)

# Optional but strongly recommended:
# keep a smaller subset while debugging
# train = train.sample(1_000_000, random_state=42).reset_index(drop=True)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_safe"
)

predictor = TabularPredictor(
    label=y_col,
    problem_type="regression",
    eval_metric=smape_scorer,
    path="models/autogluon_gpu",  # new path to avoid any cached state
    verbosity=3,                  # will log "Fitting X with num_gpus: 1"
)

predictor.fit(
    train_data=train,
    tuning_data=val,
    presets="best_quality",
    num_gpus=1,
    dynamic_stacking=False,
    num_bag_folds=0,
    num_stack_levels=0,
    # time_limit=5 * 60,
    # hyperparameters=hp,
    ag_args_fit={
        "ag.max_memory_usage_ratio": 2.0,  # allow up to 2x estimated memory
    },
)

lb = predictor.leaderboard(val, silent=True)
print(lb)

val_pred = predictor.predict(val.drop(columns=[y_col]))
test_pred = predictor.predict(test.drop(columns=[y_col]))

print("\nValidation metrics")
print("SMAPE:", smape(val[y_col], val_pred))
print("RMSE :", rmse(val[y_col], val_pred))
print("MAPE :", mape(val[y_col], val_pred))

print("\nTest metrics")
print("SMAPE:", smape(test[y_col], test_pred))
print("RMSE :", rmse(test[y_col], test_pred))
print("MAPE :", mape(test[y_col], test_pred))

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
GPU Count:          1
Memory Avail:       16.04 GB / 31.11 GB (51.6%)
Disk Space Avail:   33.05 GB / 475.82 GB (6.9%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'num_bag_folds': 0,
 'num_bag_sets': 1,
 'num_stack_levels': 0}
Full kwargs:
{'_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'ag.max_memory_usage_ratio': 2.0},
 'auto_stack': True,
 'calibrate': 'auto',
 'ds_args': {'clean_up_fits': True,
             'detection_time_frac': 0.25,
             'enable_ray_logging': True,
             'holdout_data': None,
             'holdout_frac': 

[50]	valid_set's l2: 115.404	valid_set's SMAPE: -0.312538
[100]	valid_set's l2: 96.3118	valid_set's SMAPE: -0.276164
[150]	valid_set's l2: 87.2443	valid_set's SMAPE: -0.249997
[200]	valid_set's l2: 82.8502	valid_set's SMAPE: -0.2339
[250]	valid_set's l2: 79.9562	valid_set's SMAPE: -0.22533
[300]	valid_set's l2: 77.899	valid_set's SMAPE: -0.219969
[350]	valid_set's l2: 76.1557	valid_set's SMAPE: -0.216095
[400]	valid_set's l2: 75.019	valid_set's SMAPE: -0.213011
[450]	valid_set's l2: 74.4234	valid_set's SMAPE: -0.212259
[500]	valid_set's l2: 74.4819	valid_set's SMAPE: -0.210147
[550]	valid_set's l2: 74.1786	valid_set's SMAPE: -0.207921
[600]	valid_set's l2: 74.6776	valid_set's SMAPE: -0.206554
[650]	valid_set's l2: 75.6422	valid_set's SMAPE: -0.206034
[700]	valid_set's l2: 75.5224	valid_set's SMAPE: -0.204296
[750]	valid_set's l2: 75.3318	valid_set's SMAPE: -0.203143
[800]	valid_set's l2: 75.4846	valid_set's SMAPE: -0.202482
[850]	valid_set's l2: 75.4881	valid_set's SMAPE: -0.201819
[90

Saving models/autogluon_gpu\models\LightGBMXT\model.pkl
Saving models/autogluon_gpu\utils\attr\LightGBMXT\y_pred_proba_val.pkl
	-0.1816	 = Validation score   (-SMAPE)
	528.68s	 = Training   runtime
	12.22s	 = Validation runtime
	24926.8	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: LightGBM ... Training model for up to 3004.01s of the 3003.98s of remaining time.
	Fitting LightGBM with 'num_gpus': 1, 'num_cpus': 6
	Training LightGBM with GPU, note that this may negatively impact model quality compared to CPU training.
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'device': 'gpu'}


[50]	valid_set's l2: 112.686	valid_set's SMAPE: -0.305783
[100]	valid_set's l2: 93.3743	valid_set's SMAPE: -0.267308
[150]	valid_set's l2: 84.6341	valid_set's SMAPE: -0.240088
[200]	valid_set's l2: 81.2115	valid_set's SMAPE: -0.225247
[250]	valid_set's l2: 79.5548	valid_set's SMAPE: -0.219221
[300]	valid_set's l2: 77.8165	valid_set's SMAPE: -0.215627
[350]	valid_set's l2: 78.1724	valid_set's SMAPE: -0.213104
[400]	valid_set's l2: 77.8764	valid_set's SMAPE: -0.211391
[450]	valid_set's l2: 77.7692	valid_set's SMAPE: -0.209319
[500]	valid_set's l2: 78.0427	valid_set's SMAPE: -0.207027
[550]	valid_set's l2: 77.6056	valid_set's SMAPE: -0.205469
[600]	valid_set's l2: 77.293	valid_set's SMAPE: -0.20425
[650]	valid_set's l2: 77.1404	valid_set's SMAPE: -0.203376
[700]	valid_set's l2: 76.7121	valid_set's SMAPE: -0.202714
[750]	valid_set's l2: 76.9865	valid_set's SMAPE: -0.201738
[800]	valid_set's l2: 76.8377	valid_set's SMAPE: -0.200719
[850]	valid_set's l2: 76.8146	valid_set's SMAPE: -0.199586


Saving models/autogluon_gpu\models\LightGBM\model.pkl
Saving models/autogluon_gpu\utils\attr\LightGBM\y_pred_proba_val.pkl
	-0.1781	 = Validation score   (-SMAPE)
	502.77s	 = Training   runtime
	10.18s	 = Validation runtime
	29902.7	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: RandomForestMSE ... Training model for up to 2490.73s of the 2490.69s of remaining time.
	Fitting RandomForestMSE with 'num_gpus': 1, 'num_cpus': 12
Saving models/autogluon_gpu\models\RandomForestMSE\model.pkl
Saving models/autogluon_gpu\utils\attr\RandomForestMSE\y_pred_proba_val.pkl
	-0.1822	 = Validation score   (-SMAPE)
	2149.34s	 = Training   runtime
	0.51s	 = Validation runtime
	595588.4	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: CatBoost ... Training model for up to 340.82s of the 340.79s of remaining time.
	Fitting CatBoost with 'num_gpus': 1, 'num_cpus': 6
	Train

0:	learn: 13.8581521	test: 17.6762255	best: 17.6762255 (0)	total: 122ms	remaining: 122ms
1:	learn: 13.6095203	test: 17.4455614	best: 17.4455614 (1)	total: 140ms	remaining: 0us
bestTest = 17.44556143
bestIteration = 1
0:	learn: 13.8581521	test: 17.6762255	best: 17.6762255 (0)	total: 15.8ms	remaining: 1.63s
20:	learn: 10.9190993	test: 14.2805371	best: 14.2805371 (20)	total: 344ms	remaining: 1.36s
40:	learn: 9.7393807	test: 12.9055707	best: 12.9055707 (40)	total: 665ms	remaining: 1.02s
60:	learn: 9.0761305	test: 12.1564514	best: 12.1564514 (60)	total: 989ms	remaining: 697ms
80:	learn: 8.6541406	test: 11.6747921	best: 11.6747921 (80)	total: 1.31s	remaining: 372ms
100:	learn: 8.3457004	test: 11.3614919	best: 11.3614919 (100)	total: 1.62s	remaining: 48.2ms
103:	learn: 8.3091765	test: 11.3078969	best: 11.3078969 (103)	total: 1.67s	remaining: 0us
bestTest = 11.30789687
bestIteration = 103


Saving models/autogluon_gpu\models\CatBoost\model.pkl
Saving models/autogluon_gpu\utils\attr\CatBoost\y_pred_proba_val.pkl
	-0.3171	 = Validation score   (-SMAPE)
	22.26s	 = Training   runtime
	0.02s	 = Validation runtime
	12649670.9	 = Inference  throughput (rows/s | 304499 batch size)
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: ExtraTreesMSE ... Training model for up to 318.51s of the 318.48s of remaining time.
	Fitting ExtraTreesMSE with 'num_gpus': 1, 'num_cpus': 12
	Time limit exceeded... Skipping ExtraTreesMSE.
Saving models/autogluon_gpu\models\trainer.pkl
Fitting model: NeuralNetFastAI ... Training model for up to 22.57s of the 22.54s of remaining time.
	Fitting NeuralNetFastAI with 'num_gpus': 1, 'num_cpus': 6
	To avoid this warning, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 2.0, set to >=2.30 to avoid the warning)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fi

                 model  score_test  score_val eval_metric  pred_time_test  \
0  WeightedEnsemble_L2   -0.170080  -0.170080       SMAPE       22.440953   
1             LightGBM   -0.178099  -0.178099       SMAPE        9.953848   
2           LightGBMXT   -0.181640  -0.181640       SMAPE       11.849250   
3      RandomForestMSE   -0.182238  -0.182238       SMAPE        0.615974   
4             CatBoost   -0.317128  -0.317128       SMAPE        0.113892   

   pred_time_val     fit_time  pred_time_test_marginal  \
0      22.926981  3181.512661                 0.021881   
1      10.182996   502.770756                 9.953848   
2      12.215726   528.679680                11.849250   
3       0.511257  2149.342887                 0.615974   
4       0.024072    22.264534                 0.113892   

   pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  \
0                0.017002           0.719338            2       True   
1               10.182996         502.770756

Loading: models/autogluon_gpu\models\LightGBM\model.pkl
Loading: models/autogluon_gpu\models\LightGBMXT\model.pkl
Loading: models/autogluon_gpu\models\RandomForestMSE\model.pkl
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl
Loading: models/autogluon_gpu\models\LightGBM\model.pkl
Loading: models/autogluon_gpu\models\LightGBMXT\model.pkl
Loading: models/autogluon_gpu\models\RandomForestMSE\model.pkl
Loading: models/autogluon_gpu\models\WeightedEnsemble_L2\model.pkl



Validation metrics
SMAPE: 0.170079746320193
RMSE : 8.851733673176753
MAPE : 470850.9136907516

Test metrics
SMAPE: 0.18581548818091947
RMSE : 9.655506892436764
MAPE : 0.3040622826932148


In [19]:
importance = predictor.feature_importance(train)
importance

These features in provided data are not utilized by the predictor and will be ignored: ['EIC-код_62Z1052783389048', 'EIC-код_62Z1142387881356', 'EIC-код_62Z3064979528589', 'EIC-код_62Z3089684577498', 'EIC-код_62Z3459585554968', 'EIC-код_62Z5197867294689', 'EIC-код_62Z5423451602138', 'EIC-код_62Z5692449931680', 'EIC-код_62Z5821574095639', 'EIC-код_62Z5987473505748', 'EIC-код_62Z661769289704V', 'EIC-код_62Z662465333459B', 'EIC-код_62Z6814717943705', 'EIC-код_62Z7227986894278', 'EIC-код_62Z7306502515032', 'EIC-код_62Z8635999535189', 'EIC-код_62Z8919117976790', 'EIC-код_nan', 'АЗС_АЗС_100', 'АЗС_АЗС_69', 'АЗС_АЗС_70', 'АЗС_АЗС_78', 'АЗС_АЗС_83', 'АЗС_АЗС_84', 'АЗС_АЗС_86', 'АЗС_АЗС_87', 'АЗС_АЗС_88', 'АЗС_АЗС_89', 'АЗС_АЗС_90', 'АЗС_АЗС_901', 'АЗС_АЗС_902', 'АЗС_АЗС_91', 'АЗС_АЗС_911', 'АЗС_АЗС_92', 'АЗС_АЗС_93', 'АЗС_АЗС_94', 'АЗС_АЗС_95', 'АЗС_АЗС_99', 'АЗС_nan', 'Тип_ОККО-LPG', 'Тип_nan', 'Область_nan', 'ОСР код_MGA-00200', 'ОСР код_MGA-00300', 'ОСР код_MGA-00400', 'ОСР код_MGA-00500', 

,importance,stddev,p_value,n,p99_high,p99_low
Hour,0.167714,0.002230,3.752539e-09,5,0.172307,0.163122
GPS-координати - Довгота,0.139004,0.003717,6.131950e-08,5,0.146658,0.131349
Тип_ОККО-комплекс,0.108189,0.005031,5.594048e-07,5,0.118547,0.097831
Тип_ОККО-трасова,0.097000,0.002287,3.705890e-08,5,0.101709,0.092291
Тип_ОККО-міська,0.086247,0.003283,2.515793e-07,5,0.093007,0.079486
...,...,...,...,...,...,...
EIC-код_62Z0679061212471,0.000006,0.000006,4.326131e-02,5,0.000020,-0.000007
EIC-код_62Z3831028766926,0.000006,0.000012,1.492977e-01,5,0.000031,-0.000018
EIC-код_62Z1011664646618,-0.000016,0.000087,6.492364e-01,5,0.000162,-0.000194
EIC-код_62Z3500904013798,-0.000028,0.000088,7.422624e-01,5,0.000154,-0.000210
